In [0]:
CATALOG    = "mvp_jogos"
SCHEMA_RAW = "bronze"
VOLUME     = "volumes_jogos"
ARQ_RAW    = "vgsales_1980_2024_raw.csv"
TABELA_RAW = "mvp_jogos.bronze.vgsales_raw"
URI_ARQ    = f"/Volumes/{CATALOG}/{SCHEMA_RAW}/{VOLUME}/{ARQ_RAW}"

LINHEAGEM = {
    "arquivo_origem": "Video Games Sales (1980-2024) - Raw.csv",
    "fonte_origem":   "Kaggle - 'Video Game Sales & Industry Data (1980 - 2024)' (Bhushan Divekar)",
    "url_fonte":      "https://www.kaggle.com/datasets/bhushandivekar/video-game-sales-and-industry-data-1980-2024",
    "licenca":        "CC0 - Public Domain (https://creativecommons.org/publicdomain/zero/1.0/)",
    "metodo_coleta":  "Download pontual de arquivo estático via portal Kaggle + upload manual ao Databricks Volume",
    "formato_origem": "CSV, 14 colunas, ~8,3 MB, 64.016 registros",
    "checksum_sha256": "33780cc77d2395d82628c6f9120cc0905355a4804d5616dee06bdae414aff0d1",
}

In [0]:
df_peek = spark.read.csv(URI_ARQ, header=True)  # sem schema: apenas inspeção
print("Colunas encontradas no arquivo:")
for i, c in enumerate(df_peek.columns):
    print(f"  {i+1:2d}. {c}")
df_peek.limit(3).display(truncate=60)

Colunas encontradas no arquivo:
   1. img
   2. title
   3. console
   4. genre
   5. publisher
   6. developer
   7. critic_score
   8. total_sales
   9. na_sales
  10. jp_sales
  11. pal_sales
  12. other_sales
  13. release_date
  14. last_update


img,title,console,genre,publisher,developer,critic_score,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update
/games/boxart/full_6510540AmericaFrontccc.jpg,Grand Theft Auto V,PS3,Action,Rockstar Games,Rockstar North,9.4,20.32,6.37,0.99,9.85,3.12,17-09-2013,null
/games/boxart/full_5563178AmericaFrontccc.jpg,Grand Theft Auto V,PS4,Action,Rockstar Games,Rockstar North,9.7,19.39,6.06,0.6,9.71,3.02,18-11-2014,03-01-2018
/games/boxart/827563ccc.jpg,Grand Theft Auto: Vice City,PS2,Action,Rockstar Games,Rockstar North,9.6,16.15,8.41,0.47,5.49,1.78,28-10-2002,null


In [0]:
RAW_SCHEMA = """
img           STRING,
title         STRING,
console       STRING,
genre         STRING,
publisher     STRING,
developer     STRING,
critic_score  STRING,
total_sales   STRING,
na_sales      STRING,
jp_sales      STRING,
pal_sales     STRING,
other_sales   STRING,
release_date  STRING,
last_update   STRING
"""

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df_raw = spark.read.csv(URI_ARQ, header=True, schema=RAW_SCHEMA)

df_bronze = df_raw
for nome_coluna, valor in LINHEAGEM.items():
    df_bronze = df_bronze.withColumn(nome_coluna, lit(valor))
df_bronze = df_bronze.withColumn("data_ingestao", current_timestamp())

df_bronze.write.format("delta").mode("overwrite").saveAsTable(TABELA_RAW)
print(f"\n✅ Persistida a tabela Delta: {TABELA_RAW}")


✅ Persistida a tabela Delta: mvp_jogos.bronze.vgsales_raw


In [0]:
df_bronze.limit(5).display(truncate=50)

spark.sql(f"""
    SELECT count(*)                     AS total_linhas,
           count(DISTINCT title)        AS jogos_unicos,
           count(DISTINCT console)      AS consoles_distintos,
           count(DISTINCT genre)        AS generos_distintos,
           count(DISTINCT publisher)    AS publishers_distintos
    FROM {TABELA_RAW}
""").display()

img,title,console,genre,publisher,developer,critic_score,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update,arquivo_origem,fonte_origem,url_fonte,licenca,metodo_coleta,formato_origem,checksum_sha256,data_ingestao
/games/boxart/full_6510540AmericaFrontccc.jpg,Grand Theft Auto V,PS3,Action,Rockstar Games,Rockstar North,9.4,20.32,6.37,0.99,9.85,3.12,17-09-2013,null,Video Games Sales (1980-2024) - Raw.csv,Kaggle - 'Video Game Sales & Industry Data (1980 - 2024)' (Bhushan Divekar),https://www.kaggle.com/datasets/bhushandivekar/video-game-sales-and-industry-data-1980-2024,CC0 - Public Domain (https://creativecommons.org/publicdomain/zero/1.0/),Download pontual de arquivo estático via portal Kaggle + upload manual ao Databricks Volume,"CSV, 14 colunas, ~8,3 MB, 64.016 registros",33780cc77d2395d82628c6f9120cc0905355a4804d5616dee06bdae414aff0d1,2026-09-23T18:54:56.162Z
/games/boxart/full_5563178AmericaFrontccc.jpg,Grand Theft Auto V,PS4,Action,Rockstar Games,Rockstar North,9.7,19.39,6.06,0.6,9.71,3.02,18-11-2014,03-01-2018,Video Games Sales (1980-2024) - Raw.csv,Kaggle - 'Video Game Sales & Industry Data (1980 - 2024)' (Bhushan Divekar),https://www.kaggle.com/datasets/bhushandivekar/video-game-sales-and-industry-data-1980-2024,CC0 - Public Domain (https://creativecommons.org/publicdomain/zero/1.0/),Download pontual de arquivo estático via portal Kaggle + upload manual ao Databricks Volume,"CSV, 14 colunas, ~8,3 MB, 64.016 registros",33780cc77d2395d82628c6f9120cc0905355a4804d5616dee06bdae414aff0d1,2026-09-23T18:54:56.162Z
/games/boxart/827563ccc.jpg,Grand Theft Auto: Vice City,PS2,Action,Rockstar Games,Rockstar North,9.6,16.15,8.41,0.47,5.49,1.78,28-10-2002,null,Video Games Sales (1980-2024) - Raw.csv,Kaggle - 'Video Game Sales & Industry Data (1980 - 2024)' (Bhushan Divekar),https://www.kaggle.com/datasets/bhushandivekar/video-game-sales-and-industry-data-1980-2024,CC0 - Public Domain (https://creativecommons.org/publicdomain/zero/1.0/),Download pontual de arquivo estático via portal Kaggle + upload manual ao Databricks Volume,"CSV, 14 colunas, ~8,3 MB, 64.016 registros",33780cc77d2395d82628c6f9120cc0905355a4804d5616dee06bdae414aff0d1,2026-09-23T18:54:56.162Z
/games/boxart/full_9218923AmericaFrontccc.jpg,Grand Theft Auto V,X360,Action,Rockstar Games,Rockstar North,null,15.86,9.06,0.06,5.33,1.42,17-09-2013,null,Video Games Sales (1980-2024) - Raw.csv,Kaggle - 'Video Game Sales & Industry Data (1980 - 2024)' (Bhushan Divekar),https://www.kaggle.com/datasets/bhushandivekar/video-game-sales-and-industry-data-1980-2024,CC0 - Public Domain (https://creativecommons.org/publicdomain/zero/1.0/),Download pontual de arquivo estático via portal Kaggle + upload manual ao Databricks Volume,"CSV, 14 colunas, ~8,3 MB, 64.016 registros",33780cc77d2395d82628c6f9120cc0905355a4804d5616dee06bdae414aff0d1,2026-09-23T18:54:56.162Z
/games/boxart/full_4990510AmericaFrontccc.jpg,Call of Duty: Black Ops 3,PS4,Shooter,Activision,Treyarch,8.1,15.09,6.18,0.41,6.05,2.44,06-11-2015,14-01-2018,Video Games Sales (1980-2024) - Raw.csv,Kaggle - 'Video Game Sales & Industry Data (1980 - 2024)' (Bhushan Divekar),https://www.kaggle.com/datasets/bhushandivekar/video-game-sales-and-industry-data-1980-2024,CC0 - Public Domain (https://creativecommons.org/publicdomain/zero/1.0/),Download pontual de arquivo estático via portal Kaggle + upload manual ao Databricks Volume,"CSV, 14 colunas, ~8,3 MB, 64.016 registros",33780cc77d2395d82628c6f9120cc0905355a4804d5616dee06bdae414aff0d1,2026-09-23T18:54:56.162Z


total_linhas,jogos_unicos,consoles_distintos,generos_distintos,publishers_distintos
64016,39798,81,20,3383


In [0]:
%sql
DESCRIBE DETAIL mvp_jogos.bronze.vgsales_raw

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,eb428d3d-07d9-422f-868c-792aee584200,mvp_jogos.bronze.vgsales_raw,null,,2026-09-23T18:50:18.707Z,2026-09-23T18:50:25.000Z,List(),List(),1,1591532,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> b88ce3b6-9a26-45f1-82e8-c87f0cd1efcc)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false
